# DAG comparison: two systems / truth-vs-learned


In [1]:
"""Setup helpers for the bnm example notebooks.

These notebooks are intentionally self-contained: they use small local
utilities (random DAG generator, simple synthetic-data sampler, basic
PC-CPDAG construction) so a reader can run them with only ``bnm`` and
its optional ``[viz]`` extra installed. In production, prefer
``dagsampler`` for DAG construction and synthetic data and ``cbcd`` for
the PC algorithm and CPDAG output.
"""

import bnm
import numpy as np
import pandas as pd
import random
from collections import deque

# Configure plotly so figures render in any notebook viewer (JupyterLab,
# classic Notebook, nbviewer, GitHub's static renderer) instead of only
# emitting a vnd.plotly.v1+json mimetype.
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"


# ---- DAG / data generation (would normally come from dagsampler) ----

def random_dag(n_nodes=40, edge_prob=0.1, seed=None):
    """Return a bnm.GraphLike random DAG with X_1..X_n nodes."""
    rng = random.Random(seed)
    names = tuple(f"X_{i+1}" for i in range(n_nodes))
    topo = list(range(n_nodes))
    rng.shuffle(topo)
    endpoints = np.zeros((n_nodes, n_nodes), dtype=np.int8)
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if rng.random() < edge_prob:
                src, dst = topo[i], topo[j]
                endpoints[src, dst] = bnm.EndpointMark.ARROW
                endpoints[dst, src] = bnm.EndpointMark.TAIL
    return bnm.to_graphlike(endpoints, var_names=names)


def generate_data(g, n_samples=1000, stdev=1.0, seed=None):
    """Linear-Gaussian SCM sampling. Returns a pandas DataFrame
    with columns matching g.var_names."""
    rng = np.random.default_rng(seed)
    n = g.n_vars
    names = list(g.var_names) if g.var_names else [str(i) for i in range(n)]
    arr = np.array(g.endpoints)
    children: list[list[int]] = [[] for _ in range(n)]
    in_deg = [0] * n
    for i in range(n):
        for j in range(n):
            if (
                arr[i, j] == bnm.EndpointMark.ARROW
                and arr[j, i] == bnm.EndpointMark.TAIL
            ):
                children[i].append(j)
                in_deg[j] += 1
    queue = deque(v for v in range(n) if in_deg[v] == 0)
    topo: list[int] = []
    while queue:
        v = queue.popleft()
        topo.append(v)
        for c in children[v]:
            in_deg[c] -= 1
            if in_deg[c] == 0:
                queue.append(c)
    data = pd.DataFrame(index=range(n_samples), columns=names, dtype=float)
    for v in topo:
        parents = [
            i for i in range(n)
            if arr[i, v] == bnm.EndpointMark.ARROW
            and arr[v, i] == bnm.EndpointMark.TAIL
        ]
        if not parents:
            data[names[v]] = rng.standard_normal(n_samples)
        else:
            weights = rng.uniform(0.5, 1.5, size=len(parents))
            x = rng.normal(0, stdev, size=n_samples)
            for p, w in zip(parents, weights):
                x = x + w * data[names[p]].values
            data[names[v]] = x
    return data


# ---- adjacency-matrix helpers ---------------------------------------

def from_01_adj(adj, var_names=None):
    """Convert a {0,1} adjacency matrix (1 = directed edge i→j) to a
    bnm.GraphLike. If both adj[i,j] == 1 and adj[j,i] == 1, the edge
    is treated as undirected (CPDAG-style)."""
    a = np.asarray(adj)
    n = a.shape[0]
    endpoints = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(i + 1, n):
            ij, ji = int(a[i, j]), int(a[j, i])
            if ij == 1 and ji == 1:
                endpoints[i, j] = endpoints[j, i] = bnm.EndpointMark.TAIL
            elif ij == 1:
                endpoints[i, j] = bnm.EndpointMark.ARROW
                endpoints[j, i] = bnm.EndpointMark.TAIL
            elif ji == 1:
                endpoints[i, j] = bnm.EndpointMark.TAIL
                endpoints[j, i] = bnm.EndpointMark.ARROW
    names = tuple(var_names) if var_names is not None else None
    return bnm.to_graphlike(endpoints, var_names=names)


def dag_to_cpdag(g):
    """Convert a DAG to its CPDAG (without Meek-rule closure — same
    semantics as 0.1.x's `dag_to_cpdag`). For a Meek-closed CPDAG,
    use cbcd's `DAG.to_cpdag()` instead.
    """
    g = bnm.to_graphlike(g)
    n = g.n_vars
    arr = np.array(g.endpoints)
    in_collider = np.zeros((n, n), dtype=bool)
    for v in range(n):
        parents = [
            u for u in range(n)
            if arr[u, v] == bnm.EndpointMark.ARROW
            and arr[v, u] == bnm.EndpointMark.TAIL
        ]
        for ip in range(len(parents)):
            for jp in range(ip + 1, len(parents)):
                u, w = parents[ip], parents[jp]
                if (
                    arr[u, w] == bnm.EndpointMark.NO_EDGE
                    and arr[w, u] == bnm.EndpointMark.NO_EDGE
                ):
                    in_collider[u, v] = True
                    in_collider[w, v] = True
    cpdag = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if (
                arr[i, j] == bnm.EndpointMark.ARROW
                and arr[j, i] == bnm.EndpointMark.TAIL
            ):
                if in_collider[i, j]:
                    cpdag[i, j] = bnm.EndpointMark.ARROW
                    cpdag[j, i] = bnm.EndpointMark.TAIL
                else:
                    cpdag[i, j] = bnm.EndpointMark.TAIL
                    cpdag[j, i] = bnm.EndpointMark.TAIL
    return bnm.to_graphlike(cpdag, var_names=g.var_names)


# ---- perturbation helper for "fake learned graph" demos ----------

def perturb(g, n_drops=0, n_adds=0, n_reverses=0, seed=None):
    """Return a perturbed copy of g: drop / add / reverse the requested
    number of directed edges. Used as a stand-in for a real causal-
    discovery output when we don't want to depend on a PC algorithm.

    For real workflows, swap this for `cbcd.pc(data, ...)` or any
    PC implementation whose output you can wrap in `from_01_adj()`.
    """
    g = bnm.to_graphlike(g)
    rng = random.Random(seed)
    arr = np.array(g.endpoints).copy()
    n = arr.shape[0]
    directed_edges = [
        (i, j) for i in range(n) for j in range(n)
        if arr[i, j] == bnm.EndpointMark.ARROW
        and arr[j, i] == bnm.EndpointMark.TAIL
    ]
    rng.shuffle(directed_edges)
    # Drops
    for k in range(min(n_drops, len(directed_edges))):
        i, j = directed_edges[k]
        arr[i, j] = arr[j, i] = bnm.EndpointMark.NO_EDGE
    remaining = directed_edges[n_drops:]
    # Reverses
    for k in range(min(n_reverses, len(remaining))):
        i, j = remaining[k]
        arr[i, j] = bnm.EndpointMark.TAIL
        arr[j, i] = bnm.EndpointMark.ARROW
    # Adds (only between currently-non-adjacent pairs to keep it a DAG-ish)
    non_edges = [
        (i, j) for i in range(n) for j in range(i + 1, n)
        if arr[i, j] == bnm.EndpointMark.NO_EDGE
    ]
    rng.shuffle(non_edges)
    for k in range(min(n_adds, len(non_edges))):
        i, j = non_edges[k]
        # Random direction
        if rng.random() < 0.5:
            arr[i, j] = bnm.EndpointMark.ARROW
            arr[j, i] = bnm.EndpointMark.TAIL
        else:
            arr[i, j] = bnm.EndpointMark.TAIL
            arr[j, i] = bnm.EndpointMark.ARROW
    return bnm.to_graphlike(arr, var_names=g.var_names)


# ---- viz helper: 0.1.x's compare_two_bn(option=2) ----------------

def mb_pair(g1, g2, var):
    """Return ``(g1.MB(var), g2 restricted to g1.MB(var) indices)``.
    Useful for side-by-side rendering of "true MB" vs "estimated MB
    over the same nodes." Replaces 0.1.x's ``compare_two_bn(option=2)``."""
    g1n, g2n = bnm.to_graphlike(g1), bnm.to_graphlike(g2)
    idx = bnm.markov_blanket_indices(g1n, var)
    sub_g1 = bnm.markov_blanket(g1n, var)
    arr2 = g2n.endpoints[np.ix_(idx, idx)]
    sub_names = (
        tuple(g2n.var_names[i] for i in idx) if g2n.var_names else None
    )
    sub_g2 = bnm.to_graphlike(arr2, var_names=sub_names)
    return sub_g1, sub_g2


print("bnm:", bnm.__version__)


bnm: 0.2.0.dev0


## Case 1 — condition-based comparison of two systems

Suppose we have two DAGs representing the same set of variables under different conditions, and we want to know how they differ structurally.

### Generate two related DAGs

We start from a common 100-node DAG and create two structural variants by perturbing it (drops, adds, reversals). This guarantees shared structure to compare against — useful for the per-node drill-down below.

In [2]:
base = random_dag(n_nodes=100, edge_prob=0.03, seed=55)
system1 = perturb(base, n_drops=8, n_adds=4, n_reverses=3, seed=11)
system2 = perturb(base, n_drops=12, n_adds=4, n_reverses=5, seed=22)

### Compare complexity (descriptive metrics)

`bnm.compare(g1, g2, ...)` returns a `Comparison` dataclass; `bnm.to_dataframe(c)` renders it as a wide table. The `_base` suffix marks g1's value; the unsuffixed column is g2's.

In [3]:
c = bnm.compare(
    system1, system2,
    descriptive=['n_edges', 'n_colliders', 'n_isolated_nodes'],
    comparative=None,
)
bnm.to_dataframe(c)

,node_name,n_edges_base,n_colliders_base,n_isolated_nodes_base,n_edges,n_colliders,n_isolated_nodes
0,All,134.0,125.0,4.0,130.0,108.0,7.0


### Similarity (comparative metrics)

Comparative metrics (SHD, HD, F1, TP/FP/FN, etc.) measure how alike the two systems are.

In [4]:
c = bnm.compare(
    system1, system2,
    descriptive=None,
    comparative='all',
)
bnm.to_dataframe(c).T

,0
node_name,All
additions,12.0
deletions,16.0
reversals,6.0
shd,34.0
hd,28.0
tp,112.0
fp,18.0
fn,22.0
precision,0.861538


### Per-MB drill-down

Now we ask: which Markov blankets contain at least one true-positive edge? `per_node=True` computes the per-MB sub-table.

In [5]:
c = bnm.compare(system1, system2, descriptive=None, per_node=True)
df = bnm.to_dataframe(c)
df.query("node_name != 'All' and tp > 0")[['node_name', 'tp']]

,node_name,tp
1,X_1,14.0
2,X_2,2.0
3,X_3,12.0
4,X_4,12.0
5,X_5,9.0
...,...,...
96,X_96,1.0
97,X_97,2.0
98,X_98,2.0
99,X_99,6.0


### Visual inspection of nodes of interest

For each node where the MBs share an edge, render side-by-side. The matching edge is highlighted in crimson; the anchor node is highlighted in light green.

In [6]:
from IPython.display import display

for var in df.query("node_name != 'All' and tp > 0").node_name.tolist()[:4]:
    sub1, sub2 = mb_pair(system1, system2, var)
    display(
        bnm.plot_side_by_side(
            sub1, sub2,
            name1=f'system1 MB of {var}',
            name2=f'system2 over the same nodes',
            highlight_nodes=[var],
            direction='auto',
        )
    )

## Case 2 — validating a learned DAG against the truth

Generate a true DAG, fabricate a 'learned' graph by perturbing it (a stand-in for a real causal discovery output), and compare the truth's CPDAG to the learned one. For a real PC run, swap `perturb(...)` for `cbcd.pc(data, ...)` (the suite's PC implementation) — its output already conforms to `bnm.GraphLike`.

In [7]:
true_dag = random_dag(n_nodes=40, edge_prob=0.1, seed=77)
f'true DAG: {true_dag.n_vars} variables, {bnm.count_edges(true_dag)} edges'

'true DAG: 40 variables, 64 edges'

### Construct a 'learned' graph

Drop a few edges, reverse a couple, add some noise. The result is a structurally plausible PC-output stand-in.

In [8]:
learned = perturb(true_dag, n_drops=15, n_reverses=10, n_adds=8, seed=42)
f'learned: {bnm.count_edges(learned)} edges'

'learned: 57 edges'

### Compare complexity

In [9]:
true_cpdag = dag_to_cpdag(true_dag)
c = bnm.compare(
    true_cpdag, learned,
    descriptive=['n_edges', 'n_colliders', 'n_isolated_nodes'],
    comparative=None,
)
bnm.to_dataframe(c)

,node_name,n_edges_base,n_colliders_base,n_isolated_nodes_base,n_edges,n_colliders,n_isolated_nodes
0,All,64.0,67.0,2.0,57.0,39.0,3.0


### Similarity

In [10]:
c = bnm.compare(true_cpdag, learned, descriptive=None, comparative='all')
bnm.to_dataframe(c).T

,0
node_name,All
additions,8.0
deletions,15.0
reversals,17.0
shd,40.0
hd,23.0
tp,32.0
fp,25.0
fn,32.0
precision,0.561404


### Visualize MBs that match perfectly

Find variables where F1 = 1 (the local structure is exactly right).

In [11]:
c = bnm.compare(true_cpdag, learned, descriptive=None, per_node=True)
df = bnm.to_dataframe(c)
perfect = df.query("node_name != 'All' and f1 == 1.0").node_name.tolist()
print(f'{len(perfect)} variables have a perfectly-recovered MB')
perfect[:5]

1 variables have a perfectly-recovered MB


['X_8']

Pick a partially-recovered MB (0.5 < F1 < 1) and inspect it.

In [12]:
partial = df.query("node_name != 'All' and 0.5 < f1 < 1.0")[['node_name', 'f1', 'shd']].sort_values('f1', ascending=False)
partial.head()

,node_name,f1,shd
37,X_37,0.909091,1.0
24,X_24,0.900000,2.0
16,X_16,0.888889,1.0
39,X_39,0.875000,2.0
20,X_20,0.857143,3.0


In [13]:
from IPython.display import display

if len(partial):
    var = partial.iloc[0].node_name
    sub1, sub2 = mb_pair(true_cpdag, learned, var)
    display(
        bnm.plot_side_by_side(
            sub1, sub2,
            name1=f'true MB of {var}',
            name2=f'learned MB of {var}',
            highlight_nodes=[var],
            direction='auto',
        )
    )
else:
    print('No partial-recovery MBs in this run.')

## Conclusions

Two cases for DAG comparison:

- **Side-by-side condition comparison** — build per-node drill-downs from `bnm.compare(..., per_node=True)`.
- **Validation of learned DAG** — same flow, with the truth's CPDAG as the reference and the PC output as the estimate.

All viz accepts `save=path`.

### Other Cases:
- [Evaluate Single DAG](./evaluate_single_dag.ipynb)
- [Compare Algorithms](./compare_algorithms.ipynb)
- [SID](./sid.ipynb)